# Going further: optical receivers, BER and sensitivity {#sec-receivers}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/receivers.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/receivers.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter5/detlib.py', 'detlib.py')
else:
    sys.path.insert(0, '..')                 # detlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import detlib as dl
dl.use_style()

*This chapter is not in the slides. It connects the noise analysis of @sec-noise and @sec-apd-noise with the bit-error rate of
Ch. 2 and the optical amplifiers of Ch. 4, and answers the question every link designer asks: how many photons does one bit need?*

## Key relations

A digital receiver decides, bit by bit, whether a "1" or a "0" was sent by comparing the photocurrent with a threshold. With
on–off keying (OOK) the "1" gives a mean current $I_1$ with noise $\sigma_1$ and the "0" gives $I_0$ with noise $\sigma_0$. For
Gaussian noise and the optimum threshold, the **bit-error rate** depends only on the **Q factor** (Ch. 2):

$$
Q=\frac{I_1-I_0}{\sigma_1+\sigma_0},\qquad \text{BER}=\frac12\operatorname{erfc}\!\left(\frac{Q}{\sqrt2}\right),\qquad
\text{BER}=10^{-9}\Leftrightarrow Q=6,\quad 10^{-12}\Leftrightarrow Q=7.03 .
$$

The **sensitivity** is the average optical power $\bar P=(P_1+P_0)/2$ that gives the target BER. With $P_0=0$, a photodiode of
responsivity $\mathcal R$ and an electrical bandwidth $B_e\approx R_b/2$ for a bit rate $R_b$:

- **pin, thermal-noise limited** ($\sigma_0=\sigma_1=\sigma_T$, the noise of the load or TIA): $\bar P=Q\sigma_T/\mathcal R$. It grows as
  $\sqrt{B_e}$ and is independent of the photon energy.
- **pin, shot-noise limited** (ideal amplifier): $\bar P=Q^2eB_e/\mathcal R$, i.e. $Q^2/2\approx18$ photons per bit for $\eta=1$
  at BER $10^{-9}$ (with $B_e=R_b/2$).
- **APD**: the signal is multiplied by $M$, the thermal noise is not, and the shot noise gets the excess factor $F$; there is an optimum
  $M$ (@sec-apd-noise).
- **Optically preamplified** (EDFA + pin, Ch. 4): the amplifier gain $G$ lifts the signal far above the thermal noise; the noise is now the
  beat between the signal and the amplified spontaneous emission, $\sigma^2_{s\text{-}sp}=4\mathcal R^2GP_1S_{sp}B_e$, with
  $S_{sp}=n_{sp}(G-1)h\nu$, plus the ASE–ASE beat in the optical bandwidth $B_o$.
- **Quantum limit**: an ideal photon counter makes an error only when a "1" produces no photoelectron at all, with probability
  $e^{-N_1}$. $\text{BER}=\frac12e^{-N_1}$ gives $N_1=20$ photons in a "1", i.e. **10 photons per bit** on average at BER $10^{-9}$.

## Live panel: sensitivity of four receivers

```{ojs}
//| echo: false
import {berQ, qe, h, c0, mcintyre} from "./det_sim.js"
viewof rv = Inputs.form({
  ber: Inputs.select(new Map([["10⁻³ (with FEC)", 1e-3], ["10⁻⁹", 1e-9], ["10⁻¹²", 1e-12]]), {value: 1e-9, label: "target BER"}),
  wl: Inputs.range([0.8, 1.6], {value: 1.55, step: 0.01, label: "wavelength (µm)"}),
  eta: Inputs.range([0.3, 1], {value: 0.72, step: 0.01, label: "detector QE"}),
  inoise: Inputs.range([0.5, 50], {value: 10, step: 0.5, label: "amplifier input noise (pA/√Hz)"}),
  k: Inputs.range([0, 1], {value: 0.5, step: 0.01, label: "APD ionisation ratio k"}),
  NF: Inputs.range([3, 8], {value: 5, step: 0.1, label: "EDFA noise figure (dB)"}),
  Bo: Inputs.range([12.5, 200], {value: 50, step: 2.5, label: "optical filter bandwidth (GHz)"}),
  Rb: Inputs.range([0.1, 100], {value: 10, step: 0.1, label: "bit rate (Gb/s), B_e = R_b/2"})
})
```

```{ojs}
//| echo: false
rvOut = {
  const Q = (() => { let lo = 0, hi = 12; for (let i = 0; i < 80; i++) { const m = (lo + hi) / 2; if (berQ(m) > rv.ber) lo = m; else hi = m; } return (lo + hi) / 2; })();
  const wl = rv.wl * 1e-6, hv = h * c0 / wl, R = rv.eta * qe / hv, G = 1000;
  const sens = (Rb) => {
    const Be = Rb / 2, sT = rv.inoise * 1e-12 * Math.sqrt(Be);
    const pin = (2 * qe * Be * Q * Q + 2 * Q * sT) / (2 * R);
    let apd = Infinity, Mopt = 1;
    for (let i = 0; i <= 300; i++) { const M = 10 ** (2.5 * i / 300), F = mcintyre(M, rv.k), p = (Q * Q * 2 * qe * F * M * Be + 2 * Q * sT) / (M * R) / 2; if (p < apd) { apd = p; Mopt = M; } }
    const nsp = 10 ** (rv.NF / 10) * G / (2 * (G - 1)), S = nsp * (G - 1) * hv, Bo = rv.Bo * 1e9;
    const s0 = Math.sqrt(4 * R * R * S * S * Bo * Be + sT * sT + 2 * qe * R * S * Bo * Be);
    let lo = -18, hi = -1; for (let i = 0; i < 80; i++) { const m = (lo + hi) / 2, P1 = 10 ** m, I1 = R * G * P1;
      const s1 = Math.sqrt(s0 * s0 + 4 * R * R * G * P1 * S * Be + 2 * qe * I1 * Be); if (I1 / (s1 + s0) > Q) hi = m; else lo = m; }
    const pre = 10 ** hi / 2, ql = -Math.log(2 * rv.ber) / 2 * hv * Rb;
    return {Rb, pin, apd, Mopt, pre, ql};
  };
  const sweep = d3.range(0, 61).map(i => sens(1e8 * 1000 ** (i / 60)));
  const pts = sweep.flatMap(s => [{Rb: s.Rb / 1e9, P: 10 * Math.log10(s.pin / 1e-3), c: "pin + amplifier"}, {Rb: s.Rb / 1e9, P: 10 * Math.log10(s.apd / 1e-3), c: "APD (optimum M)"},
                                   {Rb: s.Rb / 1e9, P: 10 * Math.log10(s.pre / 1e-3), c: "EDFA + pin"}, {Rb: s.Rb / 1e9, P: 10 * Math.log10(s.ql / 1e-3), c: "quantum limit"}]);
  return {Q, cur: sens(rv.Rb * 1e9), pts, hv};
}
Plot.plot({
  width: Math.min(width, 1050), height: 320, x: {label: "bit rate (Gb/s)", type: "log"}, y: {label: "sensitivity (dBm)", grid: true},
  color: {legend: true, domain: ["pin + amplifier", "APD (optimum M)", "EDFA + pin", "quantum limit"], range: ["#e34948", "#eda100", "#2a78d6", "#0b0b0b"]},
  marks: [Plot.line(rvOut.pts, {x: "Rb", y: "P", stroke: "c", strokeWidth: 2}), Plot.ruleX([rv.Rb], {stroke: "#52514e", strokeDasharray: "4,3"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">Q = ${rvOut.Q.toFixed(2)} · at ${rv.Rb} Gb/s (photons per bit = P̄/(hν R<sub>b</sub>)):<br>
pin: <b>${(10 * Math.log10(rvOut.cur.pin / 1e-3)).toFixed(1)} dBm</b> (${(rvOut.cur.pin / (rvOut.hv * rv.Rb * 1e9)).toFixed(0)} photons/bit) ·
APD at M = ${rvOut.cur.Mopt.toFixed(1)}: <b>${(10 * Math.log10(rvOut.cur.apd / 1e-3)).toFixed(1)} dBm</b> (${(rvOut.cur.apd / (rvOut.hv * rv.Rb * 1e9)).toFixed(0)}) ·
EDFA + pin: <b>${(10 * Math.log10(rvOut.cur.pre / 1e-3)).toFixed(1)} dBm</b> (${(rvOut.cur.pre / (rvOut.hv * rv.Rb * 1e9)).toFixed(0)}) ·
quantum limit ${(10 * Math.log10(rvOut.cur.ql / 1e-3)).toFixed(1)} dBm (${(-Math.log(2 * rv.ber) / 2).toFixed(1)})</div>`
```

Things to try: the pin line has slope 5 dB per decade (thermal noise ∝ $\sqrt{B_e}$), the quantum limit 10 dB per decade (fixed photons
per bit). The pin receiver is 20–35 dB above the quantum limit at all rates, because the thermal noise of even a good amplifier
corresponds to thousands of photons per bit. Lower the amplifier noise to 1 pA/√Hz: the pin improves by 10 dB.
Set $k=0$ for the APD: it gains another 9 dB, because the gain becomes almost noiseless ($F	o2$). The preamplified receiver is
within 9–14 dB of the quantum limit almost independently of the electronics; narrowing the optical filter towards the signal bandwidth
helps. Lowering the BER target to $10^{-3}$ (with forward error correction, the norm today) halves Q and saves about 3 dB for the
thermal-noise-limited pin.

## Worked comparison at 10 Gb/s, 1550 nm

In [ ]:
#| label: tbl-rx
#| tbl-cap: "Sensitivity of 10 Gb/s receivers at BER = 10⁻⁹ (Q = 6), 1550 nm, QE = 0.72 (R = 0.9 A/W), B_e = 5 GHz, amplifier noise 10 pA/√Hz, APD with k = 0.5, EDFA with G = 30 dB, NF = 5 dB, B_o = 50 GHz."
Q, Rb, wl = dl.q_from_ber(1e-9), 10e9, 1.55e-6
Be, R = Rb/2, dl.responsivity(0.72, wl)
sT = 10e-12*np.sqrt(Be)
hv = dl.h*dl.c0/wl
pin = dl.sensitivity_pin(Q, R, Be, i_amp=sT)
Ms = np.logspace(0, 2.5, 400)
apd_all = np.array([dl.sensitivity_apd(Q, R, Be, M, dl.excess_noise_mcintyre(M, 0.5), i_amp=sT) for M in Ms])
iapd = np.argmin(apd_all)
pre = dl.sensitivity_preamp(Q, R, Be, 50e9, 1000, 5.0, wl, sT)
Nql, Pql = dl.quantum_limit(1e-9, wl, Rb)
shot = Q**2*dl.qe*Be/R
rows = [("pin + TIA (thermal + shot)", pin), (f"APD, optimum M = {Ms[iapd]:.0f}", apd_all[iapd]), ("EDFA + pin", pre),
        ("ideal pin, shot noise only", shot), ("quantum limit (photon counting)", Pql)]
pd.DataFrame([(n, f"{10*np.log10(p/1e-3):.1f} dBm", f"{p*1e9:.2f} nW", f"{p/(hv*Rb):.0f}") for n, p in rows],
             columns=["receiver", "sensitivity", "P̄", "photons per bit"])

In [ ]:
#| label: fig-ber
#| fig-cap: "BER versus received power at 10 Gb/s for the four receivers of the table (computed from Q(P̄)). The curves are steep: 1 dB more power lowers the BER by orders of magnitude, the 'waterfall' of receiver measurements."
P = np.logspace(-10, -4, 400)
def q_pin(Pa): I1 = 2*R*Pa; return I1/(np.sqrt(sT**2 + 2*dl.qe*I1*Be) + sT)
def q_apd(Pa, M=Ms[iapd]): F = dl.excess_noise_mcintyre(M, 0.5); I1 = 2*M*R*Pa; return I1/(np.sqrt(sT**2 + 2*dl.qe*M**2*F*2*R*Pa*Be) + sT)
G = 1000; nsp = 10**0.5*G/(2*(G - 1)); S = nsp*(G - 1)*hv
s0 = np.sqrt(4*R**2*S**2*50e9*Be + sT**2 + 2*dl.qe*R*S*50e9*Be)
def q_pre(Pa): P1 = 2*Pa; I1 = R*G*P1; return I1/(np.sqrt(s0**2 + 4*R**2*G*P1*S*Be + 2*dl.qe*I1*Be) + s0)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for f, lab in ((q_pin, "pin"), (q_apd, "APD"), (q_pre, "EDFA + pin")):
    ax.semilogy(10*np.log10(P/1e-3), dl.ber_from_q(f(P)), label=lab)
N = P/(hv*Rb); ax.semilogy(10*np.log10(P/1e-3), 0.5*np.exp(-2*N), "k--", label="quantum limit ½exp(−2N̄)")
ax.set(xlabel="average received power (dBm)", ylabel="BER", ylim=(1e-15, 1), xlim=(-55, -10)); ax.axhline(1e-9, color=dl.INK2, ls=":", lw=1)
ax.legend(fontsize=8); plt.show()

## Why it matters: the link budget

Ch. 2 computed the attenuation of a fiber; Ch. 4 the launch power of a laser. The receiver sensitivity closes the budget:
$P_\text{launch}-\alpha L-\text{penalties}\ge\bar P_\text{sens}$. With 0 dBm launched, 0.2 dB/km and 3 dB of margin, the 10 Gb/s pin
receiver above reaches about 100 km, the APD about 150 km, and the preamplified receiver about 180 km (before dispersion, Ch. 2, becomes
the limit). Long-haul links put amplifiers every 80 km instead, and their accumulated ASE, not the receiver, sets the OSNR (Ch. 4).

::: {.callout-tip collapse="true"}
## Going further
- **Coherent detection.** Mixing the signal with a strong local oscillator $P_{LO}$ on the photodiode gives a beat current
  $2\mathcal R\sqrt{P_sP_{LO}}$ that can be made far larger than the thermal noise; the receiver becomes shot-noise limited by the LO.
  Show that the SNR then approaches $\eta P_s/(h\nu B)$ and that heterodyne PSK needs about 18 photons per bit for BER $10^{-9}$.
  Coherent receivers with balanced photodiodes and DSP carry 400–800 Gb/s per wavelength today.
- **Photon counting at the quantum limit.** Deep-space optical links (NASA's DSOC on Psyche, 2023) use superconducting nanowire
  single-photon detectors and pulse-position modulation to reach ~1 photon per bit or less. Why can PPM beat the 10 photons/bit of
  OOK?
- **Non-Gaussian statistics.** The APD gain and the ASE beat noise are not Gaussian; the Gaussian Q approximation overestimates the
  required power for APDs (McIntyre distribution) by about 1 dB.
- **Extinction ratio penalty.** If the "0" carries a power $P_0=rP_1$ (laser biased above threshold, Ch. 4), the thermal-limited
  sensitivity degrades by $(1+r)/(1-r)$. Compute the penalty for $r=0.1$.
:::

::: {.callout-note collapse="true"}
## Complements
- G. P. Agrawal, *Fiber-Optic Communication Systems* (Wiley, 4th ed.), ch. 4 (receivers) and sec. 6.5 (preamplified receivers).
- S. D. Personick, "Receiver design for digital fiber optic communication systems", *Bell Syst. Tech. J.* 52, 843 (1973).
- K. Kikuchi, "Fundamentals of coherent optical fiber communications", *J. Lightwave Technol.* 34, 157 (2016).
:::

## Try it yourself

1. Show that $Q^2eB_e/\mathcal R$ corresponds to $Q^2/2$ photons per bit when $B_e=R_b/2$ and $\eta=1$, i.e. 18 photons at BER $10^{-9}$.
   Why is it worse than the quantum limit of 10?
2. In the panel, at what bit rate do the pin and APD sensitivities become equal for $k=0.5$? Explain why APDs lose their advantage at
   low bit rates with a quiet amplifier.
3. A 25 Gb/s PON receiver must reach −28 dBm at BER $10^{-3}$. Which receiver type do you need?
4. Explain why the EDFA-preamplified receiver hardly cares about the electronic noise of the amplifier after the photodiode.